# XPert Drug Perturbation Prediction

## Overview

This notebook documents the  inference pipeline for predicting drug-induced
transcriptional perturbation (deg = X_pred − X_ctl) on a given baseline cell transcriptome.

**Pipeline**: Drug SMILES → UniMol features + HG embeddings → XPertNet ensemble → predicted deg (978 landmark genes)

## Input Parameters

| Parameter | Required | Default | Description |
|-----------|----------|----------------------|
|  | Yes | — | Root directory for this prediction task |
|  | Yes | — | Baseline transcriptome CSV (978 genes), relative to prediction_root |
|  | Yes | — | Drug list CSV ( column), relative to prediction_root |
|  | No |  | Output deg CSV path, relative to prediction_root |
|  | Yes | — | Single  or directory of  files (absolute path) |
|  | No |  | Dose index |
|  | No |  | Time index (0=3h, 1=6h, 2=24h) |
|  | No |  | Compute device |

### Dose Index Reference

| Index | Dose Range (μM) |
|-------|-----------------|
| 0 | 0.0 |
| 1 | 0.21 – 0.41 |
| 2 | 0.41 – 0.71 |
| 3 | 0.71 – 1.01 |
| 4 | 1.01 – 1.51 |
| 5 | 1.51 – 3.1 |
| 6 | 3.1 – 4.1 |
| 7 | 4.1 – 7.1 |
| 8 | 7.1 – 12.1 |
| 9 | ≥ 12.1 |

### Time Index Reference

| Index | Time |
|-------|------|
| 0 | 3h |
| 1 | 6h |
| 2 | 24h |

## FDA Drug Prediction Worked Example

This runs the full FDA drug prediction pipeline using the DTOXS warm ensemble.

**Parameters used:**
- Baseline: consensus cardiomyocyte transcriptome from 206 DTOXS control wells
- Drugs: 3127 FDA-approved drugs from ChEMBL
- Dose index: 0 (single dose in DTOXS)
- Time index: 2 (24h, only time point in DTOXS)
- Model: 7-model ensemble from 

In [ ]:
%%bash
cd /home/chang/arc/xpert
bash predict/predict.sh \
    --prediction_root predict/fda-cm \
    --ctl_csv cm_ctl_landmark.csv \
    --drug_csv fda_drugs.csv \
    --output_csv deg-perturb.csv \
    --dose_idx 0 --time_idx 2 \
    --model_path finetune/output/dtoxs_warm/ \
    --device cuda:0

## Output Format

:

| drug_name | smiles | drug_idx | A1BG | A1CF | ... | ZYX |
|-----------|--------|----------|------|------|-----|-----|
| PRAZOSIN | COc1cc2nc... | 1234 | 0.12 | -0.34 | ... | 0.05 |

- **Columns**: , ,  + 978 gene symbols
- **Values**: log2 expression change (predicted deg = X_pred − X_ctl)
- **Rows**: one per successfully predicted drug
- **Skipped drugs**: logged to  (not in SMILES index)